# 03. Preprocesamiento, Limpieza de Datos y Ajuste MINSA

**Universidad Nacional Federico Villarreal (UNFV)**  
**Facultad de Ingeniería Electrónica e Informática (FIEI)**  
**Autor:** Jimenez Villalva Franz José  
**Asignatura:** Inteligencia Artificial / Metodología de la Investigación Científica  

En este cuaderno mostramos cómo se limpian los datos crudos paso a paso: unimos las dos tablas, descartamos mediciones con error o fuera de rango, aplicamos la fórmula de corrección por altitud del MINSA y finalmente dividimos en 80/20 para entrenar sin fuga de datos.

## 1. Unión de Tablas y Filtros de Calidad
Para trabajar con datos confiables aplicamos tres filtros lógicos:
1. **Prueba de sangre válida:** Solo niños con resultado exitoso (`HC55 == 0`).
2. **Población de interés:** Menores de 3 años (entre 6 y 35 meses de edad).
3. **Rangos normales y razonables:**
   - Hemoglobina medida: entre 4.0 y 20.0 g/dL.
   - Peso: entre 3.0 y 30.0 kg.
   - Talla: entre 45.0 y 120.0 cm.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_R6 = ROOT_DIR / 'data' / 'raw' / 'RECH6_2023.csv'
RAW_R0 = ROOT_DIR / 'data' / 'raw' / 'RECH0_2023.csv'

print(':: [Paso 1/4] Cargando y uniendo tablas del INEI...')
r6 = pd.read_csv(RAW_R6, low_memory=False)
r0 = pd.read_csv(RAW_R0, low_memory=False)

r6.columns = [c.strip().replace('ï»¿', '') for c in r6.columns]
r0.columns = [c.strip().replace('ï»¿', '') for c in r0.columns]

r6['HHID'] = r6['HHID'].astype(str).str.strip()
r0['HHID'] = r0['HHID'].astype(str).str.strip()

# Cruzamos datos del menor con los datos de vivienda y altitud
merged = pd.merge(r6, r0[['HHID', 'HV040', 'HV024', 'HV025']], on='HHID', how='inner')
print(f' -> Casos combinados: {merged.shape[0]}')

# Filtramos los casos con prueba exitosa y en la edad requerida (6 a 35 meses)
df_filtro = merged[(merged['HC55'] == 0) & (merged['HC1'] >= 6) & (merged['HC1'] <= 35)].copy()

# Ajustamos escala de hemoglobina y filtramos lecturas extrañas
df_filtro['Hemoglobina_Observada'] = df_filtro['HC53'] / 10.0
df_filtro = df_filtro[(df_filtro['Hemoglobina_Observada'] >= 4.0) & (df_filtro['Hemoglobina_Observada'] <= 20.0)].copy()

# Pasamos peso a kg y talla a cm, descartando errores de tipeo en balanza
df_filtro['Peso_kg'] = pd.to_numeric(df_filtro['HC2'], errors='coerce') / 10.0
df_filtro['Talla_cm'] = pd.to_numeric(df_filtro['HC3'], errors='coerce') / 10.0
df_filtro = df_filtro.dropna(subset=['Peso_kg', 'Talla_cm'])
df_filtro = df_filtro[(df_filtro['Peso_kg'] >= 3.0) & (df_filtro['Peso_kg'] <= 30.0)]
df_filtro = df_filtro[(df_filtro['Talla_cm'] >= 45.0) & (df_filtro['Talla_cm'] <= 120.0)].copy()

print(f' -> Casos válidos con medidas coherentes: {df_filtro.shape[0]}')

## 2. Ajuste de Hemoglobina por Altitud (Norma Técnica MINSA)
En ciudades de la sierra (por encima de 1,000 msnm), el cuerpo produce más glóbulos rojos por la falta de oxígeno en la altura. Para saber si un niño tiene anemia de verdad, el MINSA exige restar un factor de corrección:

$$\text{alt} = \frac{\text{Altitud}}{1000}$$
$$\text{Factor} = -0.032 \times \text{alt} + 0.022 \times \text{alt}^2$$
$$\text{Hb}_{\text{Ajustada}} = \text{Hb}_{\text{Observada}} - \text{Factor}$$
$$\text{Anemia} = 1 \quad \text{si } \text{Hb}_{\text{Ajustada}} < 11.0 \text{ g/dL}$$

In [ ]:
print(':: [Paso 2/4] Aplicando la fórmula de ajuste por altitud del MINSA...')
df_filtro['Altitud_msnm'] = df_filtro['HV040'].astype(float)
alt_km = df_filtro['Altitud_msnm'] / 1000.0

# Si está a menos de 1000 msnm no se resta nada, si está a más se aplica el factor
factor_ajuste = np.where(
    df_filtro['Altitud_msnm'] < 1000.0,
    0.0,
    -0.032 * alt_km + 0.022 * (alt_km ** 2)
)
df_filtro['Hemoglobina_Ajustada'] = np.round(df_filtro['Hemoglobina_Observada'] - factor_ajuste, 2)
df_filtro['Anemia'] = np.where(df_filtro['Hemoglobina_Ajustada'] < 11.0, 1, 0)

# Convertimos categorías simples
df_filtro['Edad_Meses'] = df_filtro['HC1'].astype(int)
df_filtro['Sexo'] = np.where(df_filtro['HC27'] == 1, 1, 0) # 1: Varón, 0: Mujer
df_filtro['Area_Residencia'] = np.where(df_filtro['HV025'] == 2, 1, 0) # 1: Rural, 0: Urbano

print(f'Total de registros finales: {df_filtro.shape[0]} niños peruanos.')
print(f'Tasa de anemia en la muestra: {df_filtro["Anemia"].mean()*100:.2f}% ({df_filtro["Anemia"].sum()} niños con anemia)')

## 3. Partición 80/20 y Escalado sin Fuga de Datos
Separamos el 20% para el examen final del modelo (`X_test`). El escalador `StandardScaler` solo debe aprender (`fit`) de los datos de entrenamiento (`X_train`), para que el modelo no haga trampa con datos que no debería conocer.

In [ ]:
print(':: [Paso 3/4] Separando en 80/20 y escalando variables...')
features = [
    'Hemoglobina_Observada',
    'Altitud_msnm',
    'Edad_Meses',
    'Sexo',
    'Area_Residencia',
    'Peso_kg',
    'Talla_cm'
]
X = df_filtro[features]
y = df_filtro['Anemia']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Ajustamos el escalador únicamente con el conjunto de entrenamiento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Muestras para entrenar (Train): {X_train.shape[0]}')
print(f'Muestras para evaluar (Test):    {X_test.shape[0]}')

# Guardamos el escalador para que los modelos en producción usen la misma escala
models_dir = ROOT_DIR / 'models'
models_dir.mkdir(exist_ok=True)
joblib.dump(scaler, models_dir / 'scaler.joblib')
print(':: Escalador guardado en models/scaler.joblib')